In [8]:
from pathlib import Path
import re
import pandas as pd

transcript_path = Path("/Users/doniafathey/Desktop/Arabic-NLP-System/Transcripts/الساموراي  الدحيح.txt")
qa_path = Path("/Users/doniafathey/Desktop/Arabic-NLP-System/QA/3AwL93uolIA_QA.csv")

# Read raw transcript lines
with open(transcript_path, "r", encoding="utf-8") as f:
    transcript_lines_raw = f.readlines()

print("Transcript lines:", len(transcript_lines_raw))
print("First 5 raw lines:")
for l in transcript_lines_raw[:5]:
    print(l.rstrip())

# Read QA
qa = pd.read_csv(qa_path)
print("\nQA shape:", qa.shape)
print("QA columns:", qa.columns.tolist())
qa.head()

Transcript lines: 954
First 5 raw lines:
0.167: منذ زمنٍ بعيد،
2.6: في أرضٍ ليست ببعيدة...
6.818: كان هُناك طفلٌ مصريٌ سمين،
10.204: يحلم بأن يكون أول ساموراي مصري في العالم.
17.745: كَبُر الطفل،

QA shape: (300, 6)
QA columns: ['video_id', 'video_title', 'question_id', 'question', 'answer', 'difficulty']


,video_id,video_title,question_id,question,answer,difficulty
0,3AwL93uolIA,الساموراي | الدحيح,3AwL93uolIA_Q001,ماذا ورد في النص حول هذه الجزئية؟,والساموزين والسامو عليكم!,Easy
1,3AwL93uolIA,الساموراي | الدحيح,3AwL93uolIA_Q002,ما الجملة المذكورة في هذا الموضع؟,من كل مَن حرمه من حلم الساموراي...,Medium
2,3AwL93uolIA,الساموراي | الدحيح,3AwL93uolIA_Q003,كيف صيغت العبارة في النص؟,السلام عليكم ورحمة الله وبركاته،,Easy
3,3AwL93uolIA,الساموراي | الدحيح,3AwL93uolIA_Q004,ما الذي قيل في هذا السياق؟,"من برنامج ""الدحّيح""!",Medium
4,3AwL93uolIA,الساموراي | الدحيح,3AwL93uolIA_Q005,ما النص الحرفي المذكور هنا؟,خلّيني آخدك مش لمكان واحد،,Easy


In [9]:
def strip_timestamp(line: str) -> str:
    parts = line.split(":", 1)
    if len(parts) == 2:
        return parts[1].strip()
    return line.strip()

transcript_no_ts_lines = [strip_timestamp(l) for l in transcript_lines_raw]
# drop empty lines
transcript_no_ts_lines = [l for l in transcript_no_ts_lines if l]

print("First 5 lines after timestamp removal:")
for l in transcript_no_ts_lines[:5]:
    print(l)

First 5 lines after timestamp removal:
منذ زمنٍ بعيد،
في أرضٍ ليست ببعيدة...
كان هُناك طفلٌ مصريٌ سمين،
يحلم بأن يكون أول ساموراي مصري في العالم.
كَبُر الطفل،


In [10]:
AR_PUNCT_MAP = {
    "،": ",",
    "؛": ";",
    "؟": "?",
    "“": '"',
    "”": '"',
    "‘": "'",
    "’": "'",
}

def normalize_punct_and_space(text: str) -> str:
    # map punctuation
    for k, v in AR_PUNCT_MAP.items():
        text = text.replace(k, v)

    # normalize ellipsis variants
    text = re.sub(r"\.{2,}", "...", text)

    # remove weird spacing
    text = re.sub(r"\s+", " ", text).strip()

    return text

transcript_punct_lines = [normalize_punct_and_space(l) for l in transcript_no_ts_lines]
print(transcript_punct_lines[:5])

['منذ زمنٍ بعيد,', 'في أرضٍ ليست ببعيدة...', 'كان هُناك طفلٌ مصريٌ سمين,', 'يحلم بأن يكون أول ساموراي مصري في العالم.', 'كَبُر الطفل,']


In [11]:
# Arabic diacritics (tashkeel) + tatweel
DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670\u0640]")  # includes tatweel

def normalize_arabic(text: str) -> str:
    # remove diacritics/tatweel
    text = re.sub(DIACRITICS_RE, "", text)

    # normalize hamza/alef forms to bare alef
    text = re.sub(r"[إأآٱ]", "ا", text)

    # normalize ya/alef maqsura
    text = text.replace("ى", "ي")

    # normalize ta marbuta (choose ONE policy; this one maps ة→ه)
    text = text.replace("ة", "ه")

    # normalize waw/ya hamza to base letters (optional but helps consistency)
    text = text.replace("ؤ", "و").replace("ئ", "ي")

    return text

transcript_norm_lines = [normalize_arabic(l) for l in transcript_punct_lines]
print(transcript_norm_lines[:5])

['منذ زمن بعيد,', 'في ارض ليست ببعيده...', 'كان هناك طفل مصري سمين,', 'يحلم بان يكون اول ساموراي مصري في العالم.', 'كبر الطفل,']


In [12]:
def normalize_repeated_chars(text: str) -> str:
    # any char repeated 3+ times -> 2 times (e.g., "جمييييل" -> "جمييل")
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

transcript_norm_lines = [normalize_repeated_chars(l) for l in transcript_norm_lines]
print(transcript_norm_lines[:10])

['منذ زمن بعيد,', 'في ارض ليست ببعيده..', 'كان هناك طفل مصري سمين,', 'يحلم بان يكون اول ساموراي مصري في العالم.', 'كبر الطفل,', 'وسافر ل"اليابان",', 'رفضه كل معلمي الساموراي', 'والساموزين والسامو عليكم!', '"السلام عليكم"', 'تاجج الغضب بداخل الطفل..']


In [13]:
def normalize_english_case(text: str) -> str:
    # lower-case latin sequences only
    def lower_latin(m):
        return m.group(0).lower()
    return re.sub(r"[A-Za-z]+", lower_latin, text)

transcript_norm_lines = [normalize_english_case(l) for l in transcript_norm_lines]
print(transcript_norm_lines[:20])

['منذ زمن بعيد,', 'في ارض ليست ببعيده..', 'كان هناك طفل مصري سمين,', 'يحلم بان يكون اول ساموراي مصري في العالم.', 'كبر الطفل,', 'وسافر ل"اليابان",', 'رفضه كل معلمي الساموراي', 'والساموزين والسامو عليكم!', '"السلام عليكم"', 'تاجج الغضب بداخل الطفل..', 'اقسم بان ينتقم', 'من كل من حرمه من حلم الساموراي..', 'خلع رداءه,', 'وابتدا بداء جديدا..', 'وحينها, قرر..', 'انه لن ياكل العسل مره اخري..', 'بل ربما..', 'ربما.. يدس فيه شييا.', 'اعزايي المشاهدين,', 'السلام عليكم ورحمه الله وبركاته,']
